In [1]:
!nvidia-smi

Sat Jul  4 05:13:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
!pip install -q \
torch==2.8.0 \
torchvision==0.23.0 \
torchaudio==2.8.0 \
vllm==0.10.2 \
transformers==4.55.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 116.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
import torch
import vllm

print("Torch:", torch.__version__)
print("vLLM:", vllm.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Compute Capability:", torch.cuda.get_device_capability(0))

Torch: 2.8.0+cu128
vLLM: 0.10.2
CUDA available: True
GPU: Tesla T4
Compute Capability: (7, 5)


In [5]:
!pip install jedi

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [6]:
!python -m vllm.entrypoints.openai.api_server \
  --model HuggingFaceTB/SmolLM2-360M-Instruct \
  --host 0.0.0.0

2026-07-04 05:16:01.646842: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 07-04 05:16:11 [__init__.py:216] Automatically detected platform cuda.
(APIServer pid=4184) INFO 07-04 05:16:13 [api_server.py:1896] vLLM API server version 0.10.2
(APIServer pid=4184) INFO 07-04 05:16:13 [utils.py:328] non-default args: {'host': '0.0.0.0', 'model': 'HuggingFaceTB/SmolLM2-360M-Instruct'}
config.json: 100% 846/846 [00:00<00:00, 7.68MB/s]
(APIServer pid=4184) INFO 07-04 05:16:31 [__init__.py:742] Resolved architecture: LlamaForCausalLM
(APIServer pid=4184) WARNING 07-04 05:16:31 [__init__.py:2716] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
(APIServer pid=4184) WA

In [7]:
%%bash

nohup python -m vllm.entrypoints.openai.api_server \
  --model HuggingFaceTB/SmolLM2-360M-Instruct \
  --host 0.0.0.0 \
  > /tmp/vllm.log 2>&1 &

echo $! > /tmp/vllm.pid
echo "Started vLLM with PID $(cat /tmp/vllm.pid)"

Started vLLM with PID 6317


In [8]:
import time
time.sleep(60)

In [10]:
!cat /tmp/vllm.pid
!ps -p $(cat /tmp/vllm.pid) -f

6317
UID          PID    PPID  C STIME TTY          TIME CMD
root        6317       1  7 05:23 ?        00:00:16 python3 -m vllm.entrypoints.


In [11]:
!ss -tulnp | grep 8000

tcp   LISTEN 0      2048         0.0.0.0:8000       0.0.0.0:*    users:(("python3",pid=6317,fd=35))        


In [12]:
!curl http://127.0.0.1:8000/v1/models

{"object":"list","data":[{"id":"HuggingFaceTB/SmolLM2-360M-Instruct","object":"model","created":1783142909,"owned_by":"vllm","root":"HuggingFaceTB/SmolLM2-360M-Instruct","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-5019d5b7d43f4f4e8bbfb76f769847a8","object":"model_permission","created":1783142909,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [13]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:8000/v1",
)

response = client.chat.completions.create(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who are you?"}
    ],
    temperature=0.7,
    max_tokens=100,
)

print(response.choices[0].message.content)

I'm an AI assistant designed to support creative writing endeavors. I'd be happy to assist you with your writing projects, whether it's crafting a captivating story, developing a compelling character, or refining your writing style. What kind of project are you working on?
